In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar tu paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

# 03 · Preprocesamiento — de las decisiones del EDA a código con pruebas

**Proyecto RiskVision · Grupo 6.** Dataset: *Default of Credit Card Clients* (UCI, CC BY 4.0).

## Para qué sirve este notebook
El EDA (notebook 02) terminó en una tabla de decisiones. Aquí cada decisión que se puede aplicar **ya** se convierte en una **función pequeña** de `src/proyecto/data/` con **al menos una prueba** en `tests/`.
Este notebook **solo orquesta**: llama a las funciones y muestra resultados. No contiene lógica reutilizable.

Produce tres cosas: (1) datos limpios, (2) una partición entrenamiento/prueba **verificada**, y (3) la evidencia de que `data/raw/` no cambió.

**Novedad de la revisión H1 (Juan David, 2026-10-31):** antes de limpiar o partir, el código ahora **valida el contrato de los datos** y se detiene con un error claro si algo no cuadra: columnas obligatorias y sin duplicar, `ID` únicos y válidos, objetivo solo 0 y 1, números reales sin infinitos, y códigos dentro del dominio (`EDUCATION` 0–6, `MARRIAGE` 0–3, `PAY_*` de −2 a 9). No cambia ningún resultado con estos datos, pero protege cuando lleguen datos nuevos.

## Qué se hace aquí y qué se deja para después
Una regla clave del curso: **todo lo que se aprende de los datos (medias, medianas, escalas, categorías) va dentro del `Pipeline` de la etapa 4**, no aquí. Si se calculara aquí sobre todos los datos, la prueba influiría en el entrenamiento (fuga de información).

| Decisión del EDA | ¿Dónde se aplica? | Cómo |
|---|---|---|
| #1 Partir por fila y excluir `ID` del modelo | **Aquí** | `split_stratified` y `make_xy` |
| #2 Estratificar por el objetivo | **Aquí** | `split_stratified` (22,12 % de incumplimiento en cada parte) |
| #3 Agrupar códigos sin documentar y marcar `SEX`, `EDUCATION`, `MARRIAGE` como categorías | **Aquí** | `group_rare_education`, `group_rare_marriage`, `to_category` |
| #8 No borrar filas | **Aquí** | `clean` conserva todas las filas (se comprueba) |
| #4 `PAY_*` numéricas y ordenadas | **Aquí** (se comprueba que no se tocan) | Referencia provisional: la etapa 4 compara otras codificaciones (la logística las trata como estados) |
| #5 Montos sin `log(x)` | Etapa 4 | comparar `log1p` / Yeo-Johnson dentro del `Pipeline` |
| #6, #7 Con y sin `SEX` / `PAY_0` | Etapa 4 | experimentos con validación cruzada |
| #9, #10 Variables nuevas (`n_atraso`, resumen de facturas) | Etapa 4 | `04_features.ipynb` |
| Imputar, estandarizar, one-hot | Etapa 4 | dentro del `Pipeline` |

## Palabras clave
| Término | Significado simple |
|---|---|
| **Datos crudos** | El archivo original de `data/raw/`. Nunca se modifica |
| **Estratificar** | Dividir manteniendo la misma proporción de la clase rara en cada parte |
| **Conjunto de prueba** | Datos reservados que **no se usan para decidir** variables, modelo ni umbral. Ojo: ya fueron explorados (EDA) y evaluados (notebook de selección), así que **no son una evaluación final independiente** |
| **Prueba automática** | Código que falla si una regla deja de cumplirse (`pytest`) |

## 0. Cargar los datos crudos
**Qué hace la celda:** usa `load_raw()` para leer el archivo de `data/raw/` (la primera fila del `.xls` es auxiliar, por eso los nombres se leen de la segunda). Guarda una «huella» del archivo y de la tabla cargada para demostrar al final que **no se modificó nada**.

No hay datos personales (solo un `ID` anónimo), así que no hay nada que eliminar al cargar.

In [2]:
import hashlib
import pandas as pd

from proyecto.config import CATEGORICAL, ID_COL, RAW_FILE, TARGET
from proyecto.data.cleaning import clean, load_raw

df = load_raw()
print(f"{RAW_FILE.name}: {df.shape[0]:,} filas x {df.shape[1]} columnas")

# Huellas para demostrar al final que el archivo y la tabla original no cambiaron
raw_file_hash = hashlib.sha256(RAW_FILE.read_bytes()).hexdigest()
raw_table_hash = pd.util.hash_pandas_object(df, index=True).sum()

default of credit card clients.xls: 30,000 filas x 25 columnas


## 1. Limpieza
**Qué hace la celda:** aplica `clean(df)`, que reúne las reglas del EDA: agrupa los códigos sin documentar (`EDUCATION` 0, 5 y 6 → 4; `MARRIAGE` 0 → 3), marca `SEX`, `EDUCATION` y `MARRIAGE` como categorías y convierte en vacío lo imposible (no hay casos en estos datos, pero protege con datos nuevos).
Antes de limpiar, `clean` **valida** el dataset completo (ver la novedad H1 arriba). Devuelve **una copia**: `df` queda intacto. Las tablas muestran cuántos clientes había en cada código antes y después.

In [3]:
clean_df = clean(df)
print(f"antes: {df.shape}  ->  después: {clean_df.shape}")

for col in ["EDUCATION", "MARRIAGE"]:
    comparacion = pd.concat({"antes": df[col].value_counts().sort_index(),
                             "después": clean_df[col].astype(int).value_counts().sort_index()}, axis=1).fillna(0).astype(int)
    print(f"\n--- {col} (clientes por código) ---")
    display(comparacion)

antes: (30000, 25)  ->  después: (30000, 25)

--- EDUCATION (clientes por código) ---


,antes,después
EDUCATION,,
0,14,0
1,10585,10585
2,14030,14030
3,4917,4917
4,123,468
5,280,0
6,51,0



--- MARRIAGE (clientes por código) ---


,antes,después
MARRIAGE,,
0,54,0
1,13659,13659
2,15964,15964
3,323,377


**Lectura:** `EDUCATION` pasa de 7 códigos a 4 (los 345 clientes de los códigos 0, 5 y 6 se suman a «otros», que queda en 468) y `MARRIAGE` pasa de 4 a 3 (los 54 clientes del código 0 se suman a «otros», que queda en 377). Ninguna fila se borra.

## 2. Comprobaciones de la limpieza
**Qué hace la celda:** convierte cada decisión en una **afirmación** que *falla* si algo no se cumple (no es un gráfico). Si alguna es falsa, el notebook se detiene con un error que dice cuál.

In [4]:
bill_pay = [c for c in df.columns if c.startswith(("BILL_AMT", "PAY_AMT", "PAY_"))]
checks = {
    "no se perdieron filas (#8)": len(clean_df) == len(df),
    "cada cliente aparece una sola vez (ID únicos)": clean_df[ID_COL].is_unique,
    "no aparecieron vacíos nuevos": int(clean_df.isna().sum().sum()) == 0,
    "EDUCATION solo tiene los códigos 1, 2, 3 y 4 (#3)": set(clean_df["EDUCATION"].astype(int)) == {1, 2, 3, 4},
    "MARRIAGE solo tiene los códigos 1, 2 y 3 (#3)": set(clean_df["MARRIAGE"].astype(int)) == {1, 2, 3},
    "SEX, EDUCATION y MARRIAGE son categorías (#3)": all(isinstance(clean_df[c].dtype, pd.CategoricalDtype) for c in CATEGORICAL),
    "el objetivo no cambió": clean_df[TARGET].equals(df[TARGET]),
    "PAY_* y montos no se tocaron (#4, #5)": all(clean_df[c].equals(df[c]) for c in bill_pay),
    "los BILL_AMT negativos se conservaron (#5)": int((clean_df["BILL_AMT1"] < 0).sum()) == int((df["BILL_AMT1"] < 0).sum()) > 0,
    "la tabla original no se modificó": pd.util.hash_pandas_object(df, index=True).sum() == raw_table_hash,
}
result = pd.Series(checks, name="se cumple")
assert result.all(), result[~result]
print(result.to_string())

no se perdieron filas (#8)                           True
cada cliente aparece una sola vez (ID únicos)        True
no aparecieron vacíos nuevos                         True
EDUCATION solo tiene los códigos 1, 2, 3 y 4 (#3)    True
MARRIAGE solo tiene los códigos 1, 2 y 3 (#3)        True
SEX, EDUCATION y MARRIAGE son categorías (#3)        True
el objetivo no cambió                                True
PAY_* y montos no se tocaron (#4, #5)                True
los BILL_AMT negativos se conservaron (#5)           True
la tabla original no se modificó                     True


## 2b. La limpieza rechaza datos inválidos (revisión H1)
**Qué hace la celda:** toma **copias** de los datos y les introduce tres errores a propósito: un código de pago que no existe, un objetivo distinto de 0 y 1, y un `ID` repetido. `clean` debe **detenerse con un error** en los tres casos. Si no lo hiciera, el notebook falla. El original no se toca.

In [5]:
def debe_fallar(datos_malos, motivo):
    try:
        clean(datos_malos)
    except (ValueError, AssertionError) as error:
        print(f'rechazado ({motivo}): {str(error)[:90]}')
    else:
        raise AssertionError(f'clean aceptó datos inválidos: {motivo}')

debe_fallar(df.assign(PAY_0=df['PAY_0'].where(df.index != 0, 12)), 'PAY_0 = 12 fuera del dominio')
debe_fallar(df.assign(**{TARGET: df[TARGET].where(df.index != 0, 2)}), 'objetivo = 2')
debe_fallar(df.assign(ID=df['ID'].where(df.index != 1, df['ID'].iloc[0])), 'ID repetido')
assert pd.util.hash_pandas_object(df, index=True).sum() == raw_table_hash   # el original sigue igual

rechazado (PAY_0 = 12 fuera del dominio): PAY_0: hay códigos fuera del dominio permitido [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
rechazado (objetivo = 2): default payment next month: el objetivo debe contener exclusivamente 0 y 1 numéricos
rechazado (ID repetido): 1 filas repiten un ID: la partición debe hacerse por ID, no por fila


## 3. Partición entrenamiento / prueba
**Pregunta:** ¿se debe dividir por cliente o por fila?
**Mi unidad de análisis y por qué parto así:** cada fila es **un cliente distinto** (los 30.000 `ID` son únicos, comprobado en el EDA y de nuevo abajo). Como ningún cliente se repite, **partir por fila es válido**: es imposible que un mismo cliente quede en entrenamiento y en prueba. Se **estratifica** por el objetivo para que la clase rara (22,12 % incumple) tenga la misma proporción en las dos partes. Proporción de prueba: 20 %; semilla: 42.

**Qué hace la celda:** `split_stratified` parte los datos y comprueba por dentro que ningún `ID` esté en los dos lados. La tabla muestra tamaños y la tasa de incumplimiento de cada parte.

**Aviso sobre la prueba (revisión H1):** la partición es la misma del notebook de selección de modelo y esa prueba ya fue evaluada allí, además de explorada en el EDA con el dataset completo. Por eso la prueba **no es una evaluación final independiente**: a partir de aquí las decisiones se toman solo con validación dentro del entrenamiento y la prueba se reportará como evaluación complementaria, con esa limitación.

In [6]:
from proyecto.config import RANDOM_STATE, TEST_SIZE
from proyecto.data.split import assert_disjoint_ids, assert_unique_ids, split_stratified

train, test = split_stratified(clean_df, test_size=TEST_SIZE, seed=RANDOM_STATE)

resumen = pd.DataFrame({
    "filas": [len(train), len(test)],
    "% del total": [len(train) / len(clean_df), len(test) / len(clean_df)],
    "incumplen": [int(train[TARGET].sum()), int(test[TARGET].sum())],
    "tasa de incumplimiento": [train[TARGET].mean(), test[TARGET].mean()],
}, index=["entrenamiento", "prueba (reservada)"])
display(resumen.round(4))

,filas,% del total,incumplen,tasa de incumplimiento
entrenamiento,24000,0.8,5309,0.2212
prueba (reservada),6000,0.2,1327,0.2212


**Verificación de la partición** (las mismas afirmaciones que protegen `tests/test_split.py`): ninguna fila se perdió, ningún cliente está en los dos lados y la tasa de incumplimiento se mantiene.

In [7]:
assert_unique_ids(clean_df)                    # ningún cliente se repite -> partir por fila es válido
assert_disjoint_ids(train, test)               # ningún cliente en los dos lados
assert len(train) + len(test) == len(clean_df) # no se perdió ninguna fila
assert abs(train[TARGET].mean() - test[TARGET].mean()) < 0.001   # misma proporción de clase rara
print("Partición verificada: sin clientes repetidos, sin filas perdidas y con la misma tasa de incumplimiento.")

Partición verificada: sin clientes repetidos, sin filas perdidas y con la misma tasa de incumplimiento.


**Lectura:** 24.000 clientes para entrenar y 6.000 reservados, con 22,12 % de incumplimiento en las dos partes. Esta es la **misma partición** (80/20, estratificada, semilla 42) que usó el notebook de selección de modelo, así que los resultados de ambos son comparables.

## 4. Guardar los datos de la etapa
**Qué hace la celda:** guarda `train` y `test` en `data/processed/`. Esa carpeta **no se sube a GitHub**: se regenera ejecutando este notebook. El conjunto de prueba **no se usa para decidir nada** (ya estuvo expuesto, ver arriba); se reporta al final solo como evaluación complementaria.
Los archivos conservan el `ID` para poder verificar la partición; **al modelar se excluye** con `make_xy` (que quita `ID` y objetivo de los predictores).

In [8]:
import joblib
from proyecto.config import PROCESSED_DIR

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(train, PROCESSED_DIR / "train.joblib")
joblib.dump(test, PROCESSED_DIR / "test.joblib")
print("guardado en data/processed/:", sorted(p.name for p in PROCESSED_DIR.glob("*.joblib")))

from proyecto.data.split import make_xy
X_train, y_train = make_xy(train)
print(f"\npredictores para modelar: {X_train.shape[1]} columnas (sin ID ni objetivo)")
print(list(X_train.columns))

guardado en data/processed/: ['test.joblib', 'train.joblib']

predictores para modelar: 23 columnas (sin ID ni objetivo)
['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']


## 5. Las pruebas automáticas
**Qué hace la celda:** ejecuta `pytest` sobre todo el repositorio, como se haría desde la terminal (`pytest -q`). Debe existir al menos una prueba por decisión (`tests/test_cleaning.py`) y las de la partición (`tests/test_split.py`).
Para comprobar que la prueba de partición realmente protege, se verificó que **falla** si se quita la estratificación.

In [9]:
import subprocess, sys
from proyecto.config import ROOT

salida = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], cwd=ROOT, capture_output=True, text=True)
print(salida.stdout.strip().splitlines()[-1])
assert salida.returncode == 0, salida.stdout[-2000:]

142 passed, 1 warning in 16.98s


## 6. `data/raw/` no cambió
**Qué hace la celda:** compara la huella del archivo original con la que se guardó al inicio. Si alguien lo hubiera modificado, la huella sería distinta.

In [10]:
import hashlib
assert hashlib.sha256(RAW_FILE.read_bytes()).hexdigest() == raw_file_hash, "¡El archivo de data/raw cambió!"
print("data/raw/ intacto: la huella del archivo es igual a la del inicio.")

data/raw/ intacto: la huella del archivo es igual a la del inicio.


## Qué viene después (etapa 4)
Con los datos limpios y la partición reservada, el siguiente paso (`04_features.ipynb`) construye el `Pipeline` que, **aprendiendo solo del entrenamiento**, imputa con la mediana, estandariza y codifica (one-hot) las categorías. Allí también se miden las ideas de variables (utilización del crédito, rachas de atraso, `n_atraso`…) y el modelo con y sin `SEX` y con y sin `PAY_0`. Las comparaciones de transformaciones de los montos (`log1p`, Yeo-Johnson) quedan para la etapa de modelado.

## Entregable de esta etapa (hito H1)

- [x] La limpieza corre de principio a fin desde el repo (`load_raw` → `clean` → `split_stratified`).
- [x] La partición respeta mi unidad de análisis y está **verificada con una prueba**.
- [x] `pytest` en verde.
- [x] `data/raw/` no cambió (comprobado con la huella del archivo).
- [x] Declaré mi unidad de análisis por escrito (sección 3).
- [x] La limpieza rechaza datos inválidos (sección 2b, revisión H1).